# 萜合酶序列核实工具

本工具用于核实蛋白质序列是否为萜合酶（Terpene Synthase）。

**验证方法：**
1. **关键词搜索** - 在序列注释中搜索萜合酶相关关键词
2. **PROSITE模式匹配** - 匹配萜合酶保守域模式
3. **序列特征分析** - 分析序列长度、保守motif、氨基酸组成等
4. **HMMER搜索**（可选）- 使用Pfam HMM模型搜索

**输入：** FASTA格式的蛋白质序列文件
**输出：** 验证结果CSV文件，包含每条序列的验证结果和置信度

## 1. 初始化环境

In [ ]:
from protflow.utils.notebook_utils import init_notebook
from pathlib import Path
from Bio import SeqIO
import pandas as pd

# 自动初始化环境
paths = init_notebook('terpene_synthase_validation')
WORK_DIR = paths['WORK_DIR']
DATA_DIR = paths['DATA_DIR']

print(f"✓ 工作目录: {WORK_DIR}")
print(f"✓ 数据目录: {DATA_DIR}")

## 2. 导入验证模块

In [ ]:
from protflow.utils.terpene_synthase_validator import (
    validate_fasta_file,
    validate_terpene_synthase
)

print("✓ 验证模块导入成功")

## 3. 指定输入文件

请指定包含待验证序列的FASTA文件路径。

**提示：** 如果序列文件在"待办"目录中，请修改下面的路径。

In [ ]:
# 指定输入文件路径（请修改为您的实际文件路径）
# 统一使用 INPUTS_DIR，每个功能有各自的子目录
# 萜合酶验证的输入文件应放在：INPUTS_DIR / 'terpene_synthase' / 'sequences.faa'

INPUTS_DIR = paths.get('INPUTS_DIR', DATA_DIR / 'inputs')
terpene_input_dir = INPUTS_DIR / 'terpene_synthase'
terpene_input_dir.mkdir(exist_ok=True, parents=True)

input_fasta = terpene_input_dir / 'sequences.faa'  # 默认路径

# 如果文件在其他位置，可以修改为：
# input_fasta = Path('待办') / 'your_sequences.faa'  # 或其他路径

if not input_fasta.exists():
    print(f"⚠️ 输入文件不存在: {input_fasta}")
    print(f"\n请将序列文件放在以下目录：")
    print(f"  {terpene_input_dir}")
    print(f"\n或修改上面的 input_fasta 路径指向您的文件")
    print("\n正在检查可能的文件位置：")
    
    # 检查萜合酶输入目录
    if terpene_input_dir.exists():
        fasta_files = list(terpene_input_dir.glob('*.fa*'))
        if fasta_files:
            print(f"\n在 {terpene_input_dir} 中找到以下FASTA文件：")
            for f in fasta_files:
                print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")
        else:
            print(f"\n  {terpene_input_dir} 目录存在但未找到FASTA文件")
    
    # 检查待办目录（向后兼容）
    todo_dir = Path('待办')
    if todo_dir.exists():
        fasta_files = list(todo_dir.glob('*.fa*'))
        if fasta_files:
            print(f"\n在'待办'目录中找到以下FASTA文件：")
            for f in fasta_files:
                print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")
    
    # 检查其他可能的输入目录
    other_input_dirs = [
        INPUTS_DIR,
        DATA_DIR / 'inputs',
    ]
    for check_dir in other_input_dirs:
        if check_dir.exists() and check_dir != terpene_input_dir:
            fasta_files = list(check_dir.glob('*.fa*'))
            if fasta_files:
                print(f"\n在 {check_dir} 中找到以下FASTA文件：")
                for f in fasta_files[:5]:  # 只显示前5个
                    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")
                if len(fasta_files) > 5:
                    print(f"  ... 还有 {len(fasta_files) - 5} 个文件")
else:
    # 统计序列数量
    sequences = list(SeqIO.parse(input_fasta, 'fasta'))
    file_size_mb = input_fasta.stat().st_size / (1024 * 1024)
    print(f"✓ 输入文件: {input_fasta}")
    print(f"  文件大小: {file_size_mb:.2f} MB")
    print(f"  序列数量: {len(sequences)} 条")
    
    # 显示前几条序列的信息
    print(f"\n前5条序列信息：")
    for i, seq in enumerate(sequences[:5], 1):
        print(f"  {i}. {seq.id[:60]}... (长度: {len(seq.seq)} aa)")

## 4. 配置验证参数

In [ ]:
# 验证方法配置
USE_KEYWORDS = True   # 使用关键词搜索
USE_PROSITE = True    # 使用PROSITE模式匹配
USE_FEATURES = True   # 使用序列特征分析

# 输出文件配置
output_csv = WORK_DIR / 'terpene_synthase_validation_results.csv'

print("验证方法配置：")
print(f"  关键词搜索: {'启用' if USE_KEYWORDS else '禁用'}")
print(f"  PROSITE模式: {'启用' if USE_PROSITE else '禁用'}")
print(f"  序列特征分析: {'启用' if USE_FEATURES else '禁用'}")
print(f"\n输出文件: {output_csv}")

## 5. 运行验证

In [ ]:
if not input_fasta.exists():
    print("⚠️ 请先设置正确的输入文件路径（第3步）")
else:
    try:
        print("\n开始验证序列...")
        print("=" * 60)
        
        # 运行验证
        results, stats = validate_fasta_file(
            fasta_file=input_fasta,
            output_csv=output_csv,
            use_keywords=USE_KEYWORDS,
            use_prosite=USE_PROSITE,
            use_features=USE_FEATURES
        )
        
        print("\n" + "=" * 60)
        print("验证完成！")
        
    except Exception as e:
        print(f"\n✗ 验证失败: {e}")
        import traceback
        traceback.print_exc()

## 6. 查看验证结果

In [ ]:
if 'results' in locals() and 'stats' in locals():
    # 显示统计摘要
    print("=" * 60)
    print("验证结果统计")
    print("=" * 60)
    print(f"总序列数: {stats['total']}")
    print(f"\n萜合酶序列: {stats['terpene_synthase']} ({stats['terpene_synthase']/stats['total']*100:.1f}%)")
    print(f"  高置信度: {stats['high_confidence']}")
    print(f"  中置信度: {stats['medium_confidence']}")
    print(f"  低置信度: {stats['low_confidence']}")
    print(f"\n非萜合酶序列: {stats['not_terpene_synthase']} ({stats['not_terpene_synthase']/stats['total']*100:.1f}%)")
    
    # 加载结果DataFrame
    df = pd.read_csv(output_csv)
    
    # 显示高置信度的萜合酶序列
    high_conf_ts = df[(df['is_terpene_synthase'] == True) & (df['confidence'] == 'high')]
    if len(high_conf_ts) > 0:
        print(f"\n高置信度萜合酶序列（前10条）：")
        for idx, row in high_conf_ts.head(10).iterrows():
            print(f"  {row['id'][:60]}... (置信度: {row['confidence']}, 得分: {row['score']:.1f}/{row['max_score']})")
    
    # 显示非萜合酶序列（如果有）
    non_ts = df[df['is_terpene_synthase'] == False]
    if len(non_ts) > 0:
        print(f"\n非萜合酶序列（前10条）：")
        for idx, row in non_ts.head(10).iterrows():
            print(f"  {row['id'][:60]}... (得分: {row['score']:.1f}/{row['max_score']})")
    
    print(f"\n详细结果已保存到: {output_csv}")
else:
    print("⚠️ 请先运行验证（第5步）")

## 7. 详细分析（可选）

In [ ]:
if 'results' in locals():
    # 分析验证方法的效果
    print("=" * 60)
    print("验证方法效果分析")
    print("=" * 60)
    
    df = pd.read_csv(output_csv)
    import json
    
    # 统计各方法的匹配情况
    if USE_KEYWORDS:
        keyword_matches = df['methods'].apply(lambda x: json.loads(x)['keywords']['matched'] if 'keywords' in json.loads(x) else False).sum()
        print(f"关键词匹配: {keyword_matches} 条序列 ({keyword_matches/len(df)*100:.1f}%)")
    
    if USE_PROSITE:
        prosite_matches = df['methods'].apply(lambda x: json.loads(x)['prosite']['matched'] if 'prosite' in json.loads(x) else False).sum()
        print(f"PROSITE模式匹配: {prosite_matches} 条序列 ({prosite_matches/len(df)*100:.1f}%)")
    
    if USE_FEATURES:
        length_ok = df['methods'].apply(lambda x: json.loads(x)['features']['length_ok'] if 'features' in json.loads(x) else False).sum()
        ddxxd_ok = df['methods'].apply(lambda x: json.loads(x)['features']['ddxxd_motif_count'] > 0 if 'features' in json.loads(x) else False).sum()
        print(f"长度符合要求: {length_ok} 条序列 ({length_ok/len(df)*100:.1f}%)")
        print(f"包含DDXXD motif: {ddxxd_ok} 条序列 ({ddxxd_ok/len(df)*100:.1f}%)")
    
    # 显示得分分布
    print(f"\n得分分布：")
    print(df['score'].describe())
else:
    print("⚠️ 请先运行验证（第5步）")

## 8. 导出结果（可选）

In [ ]:
if 'results' in locals() and 'df' in locals():
    # 导出高置信度萜合酶序列
    high_conf_fasta = WORK_DIR / 'high_confidence_terpene_synthases.faa'
    high_conf_ids = df[(df['is_terpene_synthase'] == True) & (df['confidence'] == 'high')]['id'].tolist()
    
    if high_conf_ids:
        high_conf_seqs = [seq for seq in SeqIO.parse(input_fasta, 'fasta') if seq.id in high_conf_ids]
        SeqIO.write(high_conf_seqs, high_conf_fasta, 'fasta')
        print(f"✓ 高置信度萜合酶序列已导出到: {high_conf_fasta}")
        print(f"  共 {len(high_conf_seqs)} 条序列")
    
    # 导出所有萜合酶序列
    all_ts_fasta = WORK_DIR / 'all_terpene_synthases.faa'
    all_ts_ids = df[df['is_terpene_synthase'] == True]['id'].tolist()
    
    if all_ts_ids:
        all_ts_seqs = [seq for seq in SeqIO.parse(input_fasta, 'fasta') if seq.id in all_ts_ids]
        SeqIO.write(all_ts_seqs, all_ts_fasta, 'fasta')
        print(f"\n✓ 所有萜合酶序列已导出到: {all_ts_fasta}")
        print(f"  共 {len(all_ts_seqs)} 条序列")
    
    # 导出非萜合酶序列
    non_ts_fasta = WORK_DIR / 'non_terpene_synthases.faa'
    non_ts_ids = df[df['is_terpene_synthase'] == False]['id'].tolist()
    
    if non_ts_ids:
        non_ts_seqs = [seq for seq in SeqIO.parse(input_fasta, 'fasta') if seq.id in non_ts_ids]
        SeqIO.write(non_ts_seqs, non_ts_fasta, 'fasta')
        print(f"\n✓ 非萜合酶序列已导出到: {non_ts_fasta}")
        print(f"  共 {len(non_ts_seqs)} 条序列")
else:
    print("⚠️ 请先运行验证（第5步）")